# Notebook 01 — Data Preparation

**Goal:** Turn the 999 raw textbook chunks into labeled (query, positive, negative) training pairs.

## What happens in this notebook
1. Inspect the raw chunks from `data/processed/jurafsky_chunks.json`
2. Generate synthetic training queries (first sentence of each chunk)
3. Analyse the quality of generated pairs
4. Save `data/processed/train_pairs.json` and `data/processed/val_pairs.json`

## Why synthetic queries?
We only have 10 human-written (query, chunk) pairs in `evaluation_set.csv`.
Those 10 are precious — they're kept as the **test set** and never used for training.
To train the BiEncoder we need hundreds of labeled pairs. We generate them automatically:
- A textbook paragraph usually starts with a **topic sentence** that summarises its content.
- We use that sentence as the query. The full paragraph is the matching document.
- A random other paragraph (≥20 chunks away) is the non-matching document (negative).

---
> **Run locally or in Colab.** This notebook needs no GPU.

## Setup

In [ ]:
# ── Colab only ──────────────────────────────────────────────────────────────
import sys, os

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/Neural_Search_Engine'
    sys.path.insert(0, PROJECT_ROOT)
    os.chdir(PROJECT_ROOT)
    !pip install -q rank-bm25
else:
    PROJECT_ROOT = os.path.abspath('..')   # running from notebooks/
    sys.path.insert(0, PROJECT_ROOT)
    os.chdir(PROJECT_ROOT)

print('Working directory:', os.getcwd())
print('PROJECT_ROOT:', PROJECT_ROOT)

In [ ]:
import json
import re
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter

from src.dataset import load_chunks, generate_pairs, split_pairs, save_pairs, _extract_first_sentence

CHUNKS_PATH  = 'data/processed/jurafsky_chunks.json'
TRAIN_OUTPUT = 'data/processed/train_pairs.json'
VAL_OUTPUT   = 'data/processed/val_pairs.json'

---
## 1. Inspect the Raw Chunks

In [ ]:
chunks = load_chunks(CHUNKS_PATH)
print(f'Total chunks: {len(chunks)}')

# Word count distribution
word_counts = [c['word_count'] for c in chunks]
print(f'Word count — min: {min(word_counts)}  max: {max(word_counts)}  avg: {sum(word_counts)/len(word_counts):.0f}')

# Histogram
fig, ax = plt.subplots(figsize=(8, 3))
ax.hist(word_counts, bins=30, color='steelblue', edgecolor='white')
ax.set_xlabel('Words per chunk')
ax.set_ylabel('Count')
ax.set_title('Distribution of chunk lengths')
plt.tight_layout()
plt.show()
print('\nSample chunk:')
print(chunks[50]['content'][:400], '...')

---
## 2. Preview the Synthetic Query Extraction

In [ ]:
# Show what first-sentence extraction looks like for a sample of chunks
print(f'{"CHUNK INDEX":<12} {"EXTRACTED QUERY":<80} {"STATUS"}')
print('-' * 105)

for i in range(0, 200, 20):
    chunk = chunks[i]
    query = _extract_first_sentence(chunk['content'])
    status = '✓' if query else '✗ skipped'
    display = (query or chunk['content'])[:78]
    print(f'{i:<12} {display:<80} {status}')

---
## 3. Generate All Training Pairs

In [ ]:
# Generate (query, positive, negative) triplets
# min_distance=20 means the negative is sampled from a chunk at least 20 positions
# away from the anchor — reducing the chance of accidentally sampling a near-duplicate.

all_pairs = generate_pairs(chunks, min_distance=20, seed=42)
print(f'Generated {len(all_pairs)} pairs from {len(chunks)} chunks')
print(f'Skipped: {len(chunks) - len(all_pairs)} chunks (no usable topic sentence)')

In [ ]:
# Look at a few examples
for pair in all_pairs[10:13]:
    print('─' * 80)
    print(f"QUERY    : {pair['query']}")
    print(f"POSITIVE : [{pair['positive_id']}] {pair['positive_text'][:150]}...")
    print(f"NEGATIVE : [{pair['negative_id']}] {pair['negative_text'][:150]}...")

---
## 4. Quality Analysis

In [ ]:
# Query length distribution
query_lengths = [len(p['query'].split()) for p in all_pairs]

fig, axes = plt.subplots(1, 2, figsize=(12, 3))

axes[0].hist(query_lengths, bins=20, color='tomato', edgecolor='white')
axes[0].set_xlabel('Words in query')
axes[0].set_ylabel('Count')
axes[0].set_title('Query length distribution')

# Positive vs Negative length
pos_lengths = [len(p['positive_text'].split()) for p in all_pairs]
neg_lengths = [len(p['negative_text'].split()) for p in all_pairs]
axes[1].hist(pos_lengths, bins=20, alpha=0.6, color='steelblue', label='Positive')
axes[1].hist(neg_lengths, bins=20, alpha=0.6, color='tomato', label='Negative')
axes[1].set_xlabel('Words')
axes[1].set_title('Positive vs Negative document lengths')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f'Query words  — avg: {sum(query_lengths)/len(query_lengths):.1f}')
print(f'Positive words — avg: {sum(pos_lengths)/len(pos_lengths):.1f}')

In [ ]:
# Verify no overlap between positive and negative IDs (sanity check)
same_id = sum(1 for p in all_pairs if p['positive_id'] == p['negative_id'])
print(f'Pairs where positive == negative: {same_id}  (should be 0)')

# Check distance between positive and negative indices
id_to_idx = {c['id']: i for i, c in enumerate(chunks)}
distances = [abs(id_to_idx[p['positive_id']] - id_to_idx[p['negative_id']]) for p in all_pairs]
print(f'Min distance between pos/neg: {min(distances)}  (should be ≥ 20)')
print(f'Avg distance between pos/neg: {sum(distances)/len(distances):.0f}')

---
## 5. Train / Validation Split and Save

In [ ]:
train_pairs, val_pairs = split_pairs(all_pairs, val_size=0.10, seed=42)

print(f'Train pairs : {len(train_pairs)}')
print(f'Val pairs   : {len(val_pairs)}')
print(f'Test queries: 10  (from evaluation_set.csv — never touched during training)')

save_pairs(train_pairs, TRAIN_OUTPUT)
save_pairs(val_pairs,   VAL_OUTPUT)

In [ ]:
# Final sanity check — print first train pair
p = train_pairs[0]
print('First training pair:')
print(f"  query    : {p['query']}")
print(f"  pos_id   : {p['positive_id']}")
print(f"  neg_id   : {p['negative_id']}")
print()
print('Data preparation complete. Proceed to 02_training.ipynb')